In [151]:
import os
import PyPDF2
import numpy as np
import pandas as pd
from tabula.io import read_pdf
from datetime import datetime
import re

In [152]:
file_name = r"C:\Users\admin\Downloads\22.03.2023 £8,607.36 Bremsen Technik.pdf"

r"C:\Users\admin\Downloads\22.03.2023 £8,607.36 Bremsen Technik.pdf"

'C:\\Users\\admin\\Downloads\\22.03.2023 £8,607.36 Bremsen Technik.pdf'

In [153]:
invoice_type = "Products"

input_file = fr"C:\Users\admin\Downloads\22.03.2023 £8,607.36 Bremsen Technik.pdf"

In [154]:
table1 = read_pdf(input_file,
                  pages="1",
                  silent=True,
                  guess=False,
                  area=(150,460,186,567),
                  columns=[511,567],
                  pandas_options={'header': None},
                  encoding="windows-1254")

heading = table1[0]
display(heading)

name = "Bremsen Technik"
docnum = str(heading[1][0])
print(docnum)

date = heading[0][0]
date = str(datetime.strptime(date, "%d/%m/%y"))
print(date)

ordernum, transfernum = None, None
print(ordernum)
print(transfernum)


,0,1
0,22/03/23,36110


36110
2023-03-22 00:00:00
None
None


In [155]:
table2 = read_pdf(input_file,
                  pages="1",
                  silent=True,
                  guess=False,
                  area=(377,35,670,569),
                  columns=[228,263,320,361,406,478,569],
                  pandas_options={'header': None},
                  encoding="windows-1254")

content=table2[0]

content

,0,1,2,3,4,5,6
0,1Q19609A7 Brembo GT Kit,1.0,87083099.0,0.0,NaN,4228.0,4228.0
1,"Fluo Yellow Caliper, Drilled Disc",NaN,NaN,NaN,NaN,NaN,NaN
2,2S19001A7 Brembo GT Kit,1.0,87083099.0,0.0,NaN,2904.8,2904.8
3,"Fluo Yellow Caliper, Drilled Discs",NaN,NaN,NaN,NaN,NaN,NaN


In [156]:
content.rename(columns={
    0: 'Description',
    1: 'Qty',
    2: 'Commodity',
    3: 'Weight',
    4: 'C.O.O',
    5: 'Unit Price',
    6: 'Total'}, inplace=True)

display(content)

,Description,Qty,Commodity,Weight,C.O.O,Unit Price,Total
0,1Q19609A7 Brembo GT Kit,1.0,87083099.0,0.0,NaN,4228.0,4228.0
1,"Fluo Yellow Caliper, Drilled Disc",NaN,NaN,NaN,NaN,NaN,NaN
2,2S19001A7 Brembo GT Kit,1.0,87083099.0,0.0,NaN,2904.8,2904.8
3,"Fluo Yellow Caliper, Drilled Discs",NaN,NaN,NaN,NaN,NaN,NaN


In [157]:
#Spliting Details to SKU and Description
content[['SKU','Description1']] = content['Description'].str.split(' ', n=1, expand=True)
display(content)

,Description,Qty,Commodity,Weight,C.O.O,Unit Price,Total,SKU,Description1
0,1Q19609A7 Brembo GT Kit,1.0,87083099.0,0.0,NaN,4228.0,4228.0,1Q19609A7,Brembo GT Kit
1,"Fluo Yellow Caliper, Drilled Disc",NaN,NaN,NaN,NaN,NaN,NaN,Fluo,"Yellow Caliper, Drilled Disc"
2,2S19001A7 Brembo GT Kit,1.0,87083099.0,0.0,NaN,2904.8,2904.8,2S19001A7,Brembo GT Kit
3,"Fluo Yellow Caliper, Drilled Discs",NaN,NaN,NaN,NaN,NaN,NaN,Fluo,"Yellow Caliper, Drilled Discs"


In [158]:
content = content.dropna(subset=['Qty']).reset_index(drop=True) # Remove rows with NaN in column 2

content

,Description,Qty,Commodity,Weight,C.O.O,Unit Price,Total,SKU,Description1
0,1Q19609A7 Brembo GT Kit,1.0,87083099.0,0.0,NaN,4228.0,4228.0,1Q19609A7,Brembo GT Kit
1,2S19001A7 Brembo GT Kit,1.0,87083099.0,0.0,NaN,2904.8,2904.8,2S19001A7,Brembo GT Kit


In [159]:
dict_content = content.to_dict(orient='records')
dict_content

line_items=[]
for item in dict_content:
    # print(item)
    partNum = item['SKU']
    desc = item['Description1']
    quantity = item['Qty']
    netTotal = item['Total']

    print(partNum)

    line_item = {"line_type":"inventory",
                "sku": partNum,
                "name": desc,
                "Quantity": int(quantity),
                "net_total": float(netTotal),
                "tax_type": "INPUT2"}
    
    line_items.append(line_item)

print(line_items)

1Q19609A7
2S19001A7
[{'line_type': 'inventory', 'sku': '1Q19609A7', 'name': 'Brembo GT Kit', 'Quantity': 1, 'net_total': 4228.0, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': '2S19001A7', 'name': 'Brembo GT Kit', 'Quantity': 1, 'net_total': 2904.8, 'tax_type': 'INPUT2'}]


In [160]:
table3 = read_pdf(input_file,
                  pages="1",
                  silent=True,
                  guess=False,
                  area=(670,504,788,568),
                  columns=[568],
                  pandas_options={'header': None},
                  encoding='windows-1254')

total_content=table3[0]
display(total_content)

table3_length = len(total_content)
final_total = float(total_content[0][table3_length-1].replace(',',''))
print(final_total)

Shipping = float(total_content[0][table3_length-3])
print(Shipping)

,0
0,"7,132.80"
1,40.00
2,"1,434.56"
3,"8,607.36"


8607.36
40.0


In [161]:
#Adding shipping to the line_items
line_shipping = {"line_type":"shipping_expense",
            "sku": None,
            "name": "shipping",
            "Quantity": int(1),
            "net_total": float(Shipping),
            "tax_type": "INPUT2"}

line_items.append(line_shipping)
print(line_items)

[{'line_type': 'inventory', 'sku': '1Q19609A7', 'name': 'Brembo GT Kit', 'Quantity': 1, 'net_total': 4228.0, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': '2S19001A7', 'name': 'Brembo GT Kit', 'Quantity': 1, 'net_total': 2904.8, 'tax_type': 'INPUT2'}, {'line_type': 'Postage, Freight & Courier', 'sku': None, 'name': 'shipping', 'Quantity': 1, 'net_total': 40.0, 'tax_type': 'INPUT2'}]


In [162]:
payload = {}
keys = ["Source File",
        "Type",
        "Name",
        "Date",
        "Reference No.",
        "Order No.",
        "Transfer No.",
        "Document No.",
        "Line Items",
        "Total"]

values = [file_name,
        invoice_type,
        name,
        date,
        docnum,
        ordernum,
        transfernum,
        None,
        line_items,
        final_total]

for i, key in enumerate(keys):
    payload[key] = values[i]

payload

{'Source File': 'C:\\Users\\admin\\Downloads\\22.03.2023 £8,607.36 Bremsen Technik.pdf',
 'Type': 'Products',
 'Name': 'Bremsen Technik',
 'Date': '2023-03-22 00:00:00',
 'Reference No.': '36110',
 'Order No.': None,
 'Transfer No.': None,
 'Document No.': None,
 'Line Items': [{'line_type': 'inventory',
   'sku': '1Q19609A7',
   'name': 'Brembo GT Kit',
   'Quantity': 1,
   'net_total': 4228.0,
   'tax_type': 'INPUT2'},
  {'line_type': 'inventory',
   'sku': '2S19001A7',
   'name': 'Brembo GT Kit',
   'Quantity': 1,
   'net_total': 2904.8,
   'tax_type': 'INPUT2'},
  {'line_type': 'Postage, Freight & Courier',
   'sku': None,
   'name': 'shipping',
   'Quantity': 1,
   'net_total': 40.0,
   'tax_type': 'INPUT2'}],
 'Total': 8607.36}